# Phase 9 EVAL-06 Round-Trip Verification

Verifies the differentiable `inverse_lambert_w_transform` shipped in
`revision/core/data.py` (Phase 9 plan 01):

- **Synthetic round-trip:** max|inverse(forward(x)) − x| ≤ 1e-8 on
  `torch.randn(777, dtype=torch.float64)` (D-04).
- **Real round-trip:** max|inverse(forward(real)) − real| ≤ 1e-8 on
  the full 777-element `norm_log_delta` tensor from
  `load_and_preprocess("./data.csv")` (D-04).
- **gradcheck:** `torch.autograd.gradcheck` analytic backward vs
  finite-difference (eps=1e-6, atol=1e-6) on a small float64 sample.
- **Full pipeline round-trip (D-04b):** `full_denorm_pipeline(
  d["windowed_data"], d["transformed_norm_log_delta"], d["mu"],
  d["sigma"], d["delta"])` element-wise vs the rolling-window
  expansion of `d["log_delta"]` (length 384 × 10 = 3840) within ≤
  1e-6 tolerance, plus non-NaN gradient-flow assertion.

Output: `revision/results/eval06_roundtrip.json` with `pass: true`.

In [ ]:
import json
import os
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import torch

warnings.filterwarnings("ignore")

def _find_repo_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data.csv").exists() and (d / "revision" / "core").is_dir():
            return d
    raise FileNotFoundError(
        "Could not locate repo root from " + str(here) +
        " (looked for data.csv + revision/core)"
    )

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
from revision.core.data import (
    load_and_preprocess,
    inverse_lambert_w_transform,
    lambert_w_transform,
    full_denorm_pipeline,
    rolling_window,
)
from revision.core import WINDOW_LENGTH

d = load_and_preprocess("./data.csv")
real = d["norm_log_delta"].double()
delta_const = float(d["delta"])
print(f"Loaded: OD len={d['OD'].numel()}, log_delta len={d['log_delta'].numel()}, "
      f"windowed_data shape={tuple(d['windowed_data'].shape)}, delta={delta_const:.6f}")
assert real.numel() == 777, f"expected 777 log_delta values, got {real.numel()}"
assert d["windowed_data"].shape == (384, WINDOW_LENGTH), \
    f"expected (384, 10) windows, got {tuple(d['windowed_data'].shape)}"

In [ ]:
# (1) Synthetic round-trip (forward Lambert → inverse Lambert)
x_synth = torch.randn(777, dtype=torch.float64, requires_grad=True)
y_synth = lambert_w_transform(x_synth, delta_const)
x_synth_rt = inverse_lambert_w_transform(y_synth, delta_const)
err_synth = (x_synth_rt - x_synth).abs().max().item()
print(f"Synthetic round-trip max_abs_error = {err_synth:.3e}  (target ≤ 1e-8)")

# (2) Real round-trip on the full norm_log_delta tensor
y_real = lambert_w_transform(real, delta_const)
real_rt = inverse_lambert_w_transform(y_real, delta_const)
err_real = (real_rt - real).abs().max().item()
print(f"Real round-trip      max_abs_error = {err_real:.3e}  (target ≤ 1e-8)")

# (3) gradcheck — analytic backward vs numerical Jacobian
gradcheck_passed = torch.autograd.gradcheck(
    lambda v: inverse_lambert_w_transform(v, delta_const),
    (torch.randn(20, dtype=torch.float64, requires_grad=True),),
    eps=1e-6, atol=1e-6,
)
print(f"gradcheck passed     = {gradcheck_passed}")

# (4) TRUE full-pipeline round-trip per D-04b.
# full_denorm_pipeline applies the inverse of the forward training-data path:
#   reshape(-1) (un-window)  →  rescale (un-min-max → pre-Lambert range)
#   →  lambert_w_transform (heavy→Gaussian, undoes the inverse-Lambert step
#      that load_and_preprocess applied at line 234)
#   →  denormalize(mu, sigma)  (undoes the standardization step)
# Therefore feeding d["windowed_data"] (the training-data tensor itself) through
# this pipeline must reproduce — element-by-element — the rolling-window
# expansion of d["log_delta"], i.e. the same un-windowed log-return sequence.
windowed = d["windowed_data"].double().requires_grad_(True)
pipe_out = full_denorm_pipeline(
    windowed,
    d["transformed_norm_log_delta"],
    d["mu"].double() if torch.is_tensor(d["mu"]) else torch.tensor(d["mu"], dtype=torch.float64),
    d["sigma"].double() if torch.is_tensor(d["sigma"]) else torch.tensor(d["sigma"], dtype=torch.float64),
    delta_const,
)
# Reference: same un-windowing applied directly to log_delta
log_delta_windowed = rolling_window(d["log_delta"].double(), WINDOW_LENGTH, 2)
log_delta_ref_flat = log_delta_windowed.reshape(-1)
assert pipe_out.shape == log_delta_ref_flat.shape, \
    f"pipe_out shape {tuple(pipe_out.shape)} != ref shape {tuple(log_delta_ref_flat.shape)}"
err_full = (pipe_out - log_delta_ref_flat).abs().max().item()
print(f"Full pipeline round-trip max_abs_error = {err_full:.3e}  (target ≤ 1e-6, D-04b)")

# Non-NaN gradient-flow assertion on the same pipeline call
pipe_out.sum().backward()
grad_finite = (
    windowed.grad is not None
    and not torch.isnan(windowed.grad).any().item()
    and not torch.isinf(windowed.grad).any().item()
)
print(f"Full pipeline grad_finite = {grad_finite}")
assert grad_finite, "full_denorm_pipeline produced NaN/inf gradient"

In [ ]:
tolerance = {
    "synthetic": 1e-8,
    "real": 1e-8,
    "full_pipeline": 1e-6,
    "gradcheck": True,
}
delta_report = {
    "synthetic": float(err_synth),
    "real": float(err_real),
    "full_pipeline": float(err_full),
    "gradcheck_passed": bool(gradcheck_passed),
}
passed = (
    delta_report["synthetic"] <= tolerance["synthetic"]
    and delta_report["real"] <= tolerance["real"]
    and delta_report["full_pipeline"] <= tolerance["full_pipeline"]
    and delta_report["gradcheck_passed"]
)

def _git_sha():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return "unknown"

artifact = {
    "delta": delta_report,
    "tolerance": tolerance,
    "pass": bool(passed),
    "seed": SEED,
    "git_sha": _git_sha(),
    "notes": "Phase 9 EVAL-06: differentiable inverse Lambert W round-trip.",
}

out = Path("revision/results/eval06_roundtrip.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(artifact, indent=2))
print(json.dumps(artifact, indent=2))

assert passed, f"EVAL-06 FAILED: delta={delta_report} tolerance={tolerance}"
print("EVAL-06 round-trip PASSED")